In [ ]:
def _add_predispatch_step_features(df: pd.DataFrame) -> pd.DataFrame:
    """
    Features derived from the multi-step predispatch and PDPASA columns
    produced by Dataset/7_predispatch_source.py / Dataset/8_pdpasa_source.py.

    Column naming convention (set by prepare_region_data):
      predispatch_rrp_h{k}[T]  = AEMO pre-dispatch RRP forecast for T+k
                                  from the run issued at T
      pasa_avail_gen_h{k}[T]   = PDPASA available-generation forecast for T+k
      pasa_demand50_h{k}[T]    = PDPASA 50th-percentile demand forecast for T+k

    For each prediction step h in [FORECAST_GAP,
    FORECAST_GAP + FORECAST_HORIZON), this function adds:
      pd_step_h{h}          — raw RRP forecast ($/MWh) for that step
      pd_step_h{h}_asinh    — arcsinh(forecast / 100)
      pd_step_h{h}_spike    — binary: forecast > $300
      pasa_avail_h{h}       — PDPASA available generation for that step (if present)
      pasa_margin_h{h}      — PDPASA spare capacity (avail - demand50) for that step

    No-op if no predispatch_rrp_h{k} columns are present.
    """
    df = df.copy()
    gap     = FORECAST_GAP
    horizon = FORECAST_HORIZON
    SPIKE_THR = 300.0

    for h in range(gap, gap + horizon):
        rrp_col = f"predispatch_rrp_h{h}"
        if rrp_col not in df.columns or df[rrp_col].isna().all():
            continue

        p = df[rrp_col].astype("float64")
        df[f"pd_step_h{h}"]       = p.astype(np.float32)
        df[f"pd_step_h{h}_asinh"] = np.arcsinh(p / 100.0).astype(np.float32)
        df[f"pd_step_h{h}_spike"] = (p > SPIKE_THR).astype(np.float32)

    # PDPASA step columns for PASA available generation and margin
    for h in range(gap, gap + horizon):
        avail_col = f"pasa_avail_gen_h{h}"
        dem50_col = f"pasa_demand50_h{h}"
        if avail_col in df.columns and not df[avail_col].isna().all():
            pa = df[avail_col].astype("float64")
            df[f"pasa_avail_h{h}"] = pa.astype(np.float32)
            if dem50_col in df.columns and not df[dem50_col].isna().all():
                d50 = df[dem50_col].astype("float64")
                df[f"pasa_margin_h{h}"] = (pa - d50).clip(-5000, 10000).astype(np.float32)

    return df